In [1]:
import geopandas as gpd
import pandas as pd

# 1. Definir os caminhos dos ficheiros
gpkg_path = "MstCSCS_Sem_2526.gpkg"
consumo_path = "consumo_anual_estimado_2024.csv"

print("A carregar a camada de edifícios (17559 atributos)...")
# Carrega a camada específica com os códigos postais de 7 dígitos (cp7)
edificios_17k = gpd.read_file(gpkg_path, layer="postal_code_buildings_assigned")

print("A carregar e processar os dados de consumo...")
# Carrega o perfil de consumo anual horário
df_consumo = pd.read_csv(consumo_path, sep=";")

# NOTA: Como o teu ficheiro horário representa o consumo geral da região de Aveiro,
# para obteres o consumo por edifício dentro de cada CP7, primeiro calculamos 
# a área total de edifícios em cada código postal para fazermos a distribuição ponderada.

# 2. Calcular a área total de edifícios por cada Código Postal (CP7)
area_por_cp7 = edificios_17k.groupby("cp7")["building_area_m2"].sum().reset_index()
area_por_cp7 = area_por_cp7.rename(columns={"building_area_m2": "area_total_cp7"})

# Juntar a informação da área total do grupo de volta à tabela de edifícios
edificios_17k = edificios_17k.merge(area_por_cp7, on="cp7", how="left")

# 3. Calcular a energia total anual consumida em 2024
total_energia_anual = df_consumo["Energia ativa (kWh)"].sum()

# 4. Calcular o consumo por Código Postal (atribuindo uma quota do consumo total com base na relevância da área do CP7 na cidade)
area_total_cidade = edificios_17k["building_area_m2"].sum()
df_consumo_por_cp7 = area_por_cp7.copy()
df_consumo_por_cp7["energia_total_cp7"] = (df_consumo_por_cp7["area_total_cp7"] / area_total_cidade) * total_energia_anual

# 5. Juntar o consumo do CP7 aos edifícios e aplicar a distribuição final por edifício
edificios_17k = edificios_17k.merge(df_consumo_por_cp7[["cp7", "energia_total_cp7"]], on="cp7", how="left")

# Fórmula final: Consumo do Edifício = (Área do Edifício / Área Total do seu CP7) * Energia Total do seu CP7
edificios_17k["consumo_anual_estimado_kWh"] = (
    edificios_17k["building_area_m2"] / edificios_17k["area_total_cp7"]
) * edificios_17k["energia_total_cp7"]

# Limpeza de colunas auxiliares para o resultado ficar limpo
edificios_17k = edificios_17k.drop(columns=["area_total_cp7", "energia_total_cp7"])

# Verificar o resultado das primeiras linhas
print("\nCruzamento avançado por CP7 concluído com sucesso!")
print(edificios_17k[["polygon_id", "cp7", "building_area_m2", "consumo_anual_estimado_kWh"]].head())

# Guardar a nova base de dados resultante
edificios_17k.to_csv("edificios_17k_com_consumo_por_cp7.csv", index=False)
print("\nFicheiro 'edificios_17k_com_consumo_por_cp7.csv' guardado com sucesso.")

A carregar a camada de edifícios (17559 atributos)...
A carregar e processar os dados de consumo...

Cruzamento avançado por CP7 concluído com sucesso!
  polygon_id       cp7  building_area_m2  consumo_anual_estimado_kWh
0     100024  3800-200         99.080842                 5686.403214
1     100032  3810-086        373.299217                21424.221049
2     100039  3800-176         20.819647                 1194.871814
3      10004  3800-302        155.768200                 8939.778574
4      10005  3800-041        176.255125                10115.554981

Ficheiro 'edificios_17k_com_consumo_por_cp7.csv' guardado com sucesso.
